# v18b — seed stability of v18's verdict

**CPU only; Llama is not loaded.** It takes about 1 hour with everything on, or about 25 minutes with `RUN_V17 = False`.
Each seed's scores are saved as soon as they are produced, so a re-run skips what is already done.

## Why

v18 showed that fixing the PCA seed alone moved MSLSAM v17 by ~1.3 pooled points in-distribution and by ~7 points
leave-one-corpus-out. v18's key result, **H2** (hidden states beat TF-IDF on the payload-held-out split), passed
with a pooled margin of 2.88 against a 2.0 threshold. That was a **single PCA seed (42)**. Before ~7 h of GPU goes into
v19, this notebook checks whether H2 depends on that seed.

## What changes, and what does not

- **Changes:** only the PCA `random_state`. Seeds **0, 1, 2, 3, 4** are new; 42 is v18's.
- **Fixed, exactly as in v18:** the folds (P2 loaded from `results/v18/folds_P2.npy`, and P2p derived from it the same way), the
  C grid {0.001, 0.01, 0.1, 1}, the inner-CV split, the heads, and the threshold protocol (v17b's code, verbatim).
  TF-IDF has no randomness, so its v18 scores are reused.
- **Integrity first:** seed 42 is re-run on P2 and **must reproduce v18's saved scores** before any new seed runs.
- Methods: MSLSAM-P (LR) on P2, P2p and LOCO; MSLSAM v17 on P2 and LOCO (skipped if `RUN_V17 = False`).
  The HGB probe is not re-run, because it never decides anything.

## Pre-registered reading (written before any v18b number exists)

For each new seed s ∈ {0, 1, 2, 3, 4}, H2 is evaluated with **exactly v18's rule**. On P2, MSLSAM-P must be ≥ 2 pooled
points below TF-IDF, with McNemar in its favour on ≥ 2 families and against it on none. On P2p, it must not be worse than TF-IDF.

| H2 holds in … of the 5 new seeds | reading | v19 |
|---|---|---|
| **5 or 4** | **H2 is STABLE**, and v18's claim stands | **recommended** |
| 3 or 2 | H2 is **seed-dependent (fragile)**. Report it as such; v18's pass was partly luck | not justified by this evidence |
| 1 or 0 | H2 is **RETRACTED**; seed 42 was a lucky draw | not justified |

Reported as descriptive only; none of these changes the H2 reading:
- **Headline stability:** SPML and DeepSet ASR under P2 for every seed (min–max), and in how many seeds each is < 5%.
- **H1 and H3 across seeds.** If H1 holds in ≥ 4 of 5 seeds, v18's "H1 does not hold" becomes *seed-dependent*. The same applies to H3.
- **The seed spread of each method**: mean, SD and min–max of pooled ASR per protocol. It measures how unstable
  each pipeline is, and it is the noise floor for every earlier MSLSAM comparison.

Not allowed afterwards: adding seeds until a count changes, dropping a seed, or changing the C grid or the folds.

In [ ]:
import os, json, time, gc, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import binomtest
warnings.filterwarnings("ignore")

SEED, N_THRESH, FPR_BUDGET = 42, 400, 0.10   # SEED stays 42 for folds / inner CV / heads
SEEDS    = [0, 1, 2, 3, 4]    # PCA seeds -- fixed BEFORE the run; 42 (v18) is re-run for integrity only
RUN_V17  = True               # MSLSAM v17 across seeds (H1 + its own noise floor); False saves ~30 min
RUN_LOCO = True               # LOCO across seeds (H3); False saves ~10 min
PCA_DIMS, N_SPLITS = 512, 5                  # MSLSAM v17 reference -- unchanged
PROBE_DIMS, PROBE_C = 256, [0.001, 0.01, 0.1, 1.0]   # MSLSAM-P -- fixed BEFORE the run
RUN_HGB_PROBE = False                        # v18b: the HGB probe never decides -> not re-run
RES, FIG = "results/v18b", "mslsamv18b"
SC = f"{RES}/scores"; os.makedirs(SC, exist_ok=True)
CACHE = "representation_cache/joined_v16"
V17_SC = "results/v17/scores"
EMB_PATH = "results/v17/minilm_emb.npy"

man = json.load(open(f"{CACHE}/manifest.json")); LAYERS = man["layers"]
rows = pd.read_csv(f"{CACHE}/rows_v16.csv").reset_index(drop=True)
corpus = pd.read_csv("datasets/unified/injection_corpus_v15.csv").reset_index(drop=True)
assert len(rows) == len(corpus) and (rows.corpus.values == corpus.corpus.values).all() \
       and (rows.label.values == corpus.label.values).all(), "cache rows and corpus disagree"
is_base = (rows.corpus == "Baseline-Dolly").values
ev = rows[~is_base].reset_index(drop=True); ev_rows = np.where(~is_base)[0]
evc = corpus[~is_base].reset_index(drop=True)
y, cor = ev.label.values, ev.corpus.values
texts = evc.text.astype(str).tolist()
r17 = pd.read_csv(f"{V17_SC}/_rows.csv")
assert (r17.corpus.values == cor).all() and (r17.label.values == y).all(), "v17 score order differs"
S = pd.DataFrame({"row": np.arange(len(ev)), "label": y, "corpus": cor})

# ---------------- the four protocols ----------------
IND = np.isin(cor, ["BIPIA", "InjecAgent"])                 # indirect injections: payload inside a carrier
st, en = evc.span_start.fillna(-1).astype(int).values, evc.span_end.fillna(-1).astype(int).values
payload = np.array([t[a:b].strip().lower() if i else "" for t, a, b, i in zip(texts, st, en, IND)], dtype=object)
assert all(p != "" for p in payload[IND]), "an indirect injection has no payload span"
carrier = ev.pair_id.fillna("").astype(str).values
G1 = ev.group_carrier.astype(str).values
G2 = np.array([("PAY::" + p) if i else g for p, i, g in zip(payload, IND, G1)], dtype=object)
print(f"indirect injections: {IND.sum():,} rows, {len(set(payload[IND])):,} distinct payload strings "
      f"(payload_id would have said {ev.payload_id[IND].nunique():,} -- ids are NOT strings)")

def _splits_from_ids(fid):
    return [(np.where(fid != k)[0], np.where(fid == k)[0]) for k in range(fid.max() + 1)]

f1 = np.load(f"{V17_SC}/_folds_indist.npy"); assert len(f1) == len(ev) and (f1 >= 0).all()
skf = StratifiedGroupKFold(N_SPLITS, shuffle=True, random_state=SEED)
f2 = np.full(len(ev), -1)
for k, (_, te) in enumerate(skf.split(np.zeros(len(ev)), y, groups=G2)): f2[te] = k
assert (f2 >= 0).all()
assert (np.load("results/v18/folds_P2.npy") == f2).all(), "P2 folds differ from v18 -- STOP"
print("P2 folds: identical to v18 (asserted)")

def _purge(tr, te):
    # P2p: drop from TRAINING every row whose carrier appears among this test fold's indirect injections
    tc = set(carrier[te][IND[te]]); tc.discard("")
    return tr[~np.isin(carrier[tr], list(tc))], te

LOCO_FAM = {"SPML": ["SPML", "SPML-Benign"], "DeepSet": ["DeepSet", "DeepSet-Benign"],
            "BIPIA": ["BIPIA", "BIPIA-Clean", "BIPIA-HardNeg"],
            "InjecAgent": ["InjecAgent", "InjecAgent-Clean", "InjecAgent-HardNeg"]}
FAM_ROWS = np.isin(cor, sum(LOCO_FAM.values(), []))
PROTOCOLS = {
    "P1":   _splits_from_ids(f1),
    "P2":   _splits_from_ids(f2),
    "P2p":  [_purge(tr, te) for tr, te in _splits_from_ids(f2)],
    "LOCO": [(np.where(~np.isin(cor, cs))[0], np.where(np.isin(cor, cs))[0]) for cs in LOCO_FAM.values()],
}

# ---------------- leak table: what did training see about each test injection? ----------------
FAMS = ["SPML", "DeepSet", "BIPIA", "InjecAgent"]
key_txt = np.array([t.strip().lower() for t in texts], dtype=object)
LEAK = {}
print(f"\n{'protocol':<6}{'family':<12}{'payload seen':>14}{'carrier seen':>14}{'exact text seen':>17}{'train rows':>12}")
for P in ["P1", "P2", "P2p"]:
    for fam in FAMS:
        ps, cs, ts = [], [], []
        for tr, te in PROTOCOLS[P]:
            m = te[cor[te] == fam]
            key = payload if fam in ("BIPIA", "InjecAgent") else key_txt
            tr_inj = tr[y[tr] == 1]
            ps.append(np.isin(key[m], key[tr_inj]))
            cs.append(np.isin(carrier[m], carrier[tr]) & (carrier[m] != ""))
            ts.append(np.isin(key_txt[m], key_txt[tr]))
        ps, cs, ts = (np.concatenate(v).mean() * 100 for v in (ps, cs, ts))
        LEAK[f"{P}/{fam}"] = {"payload_seen": ps, "carrier_seen": cs, "text_seen": ts}
        ntr = int(np.mean([len(tr) for tr, _ in PROTOCOLS[P]]))
        print(f"{P:<6}{fam:<12}{ps:>13.1f}%{cs:>13.1f}%{ts:>16.1f}%{ntr:>12,}")
for fam in FAMS:
    assert LEAK[f"P2/{fam}"]["payload_seen"] == 0 and LEAK[f"P2p/{fam}"]["payload_seen"] == 0, f"P2 leaks payloads on {fam}"
    assert LEAK[f"P2p/{fam}"]["carrier_seen"] == 0, f"P2p leaks carriers on {fam}"
print("\nasserted: P2 and P2p hold out every test payload; P2p also holds out every test carrier.")
for tr, te in PROTOCOLS["P2p"]:
    n = lambda c: int((cor[tr] == c).sum())
    print(f"  P2p fold train: BIPIA inj {n('BIPIA'):>4} | InjecAgent inj {n('InjecAgent'):>3} | InjecAgent-Clean {n('InjecAgent-Clean'):>3}")
print("  (P2p starves InjecAgent of training data by design -- for that family it approaches LOCO)")

In [ ]:
# ---- the shared protocol: copied VERBATIM from v17b cell 3 by the generator (not retyped) ----
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k/n; den = 1 + z*z/n; c = (p + z*z/(2*n))/den
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/den
    return max(0, c-h)*100, min(1, c+h)*100

def thr(cal, budget=FPR_BUDGET):
    # TIE-AWARE threshold: the smallest cutoff whose calibration FPR is <= budget.
    # np.quantile alone breaks on tied scores. The keyword-rule score is a COUNT, and 100% of
    # SPML-Benign scores exactly 0 -- so the 90th percentile is 0, and with a '>=' rule EVERY
    # prompt is flagged: ASR 0%, FPR 100%. v17b run 1 reported that as "rules beat MSLSAM on
    # every family". With ties, step up to the next distinct value, which is <= budget by
    # construction. Continuous scores are unaffected (re-scored: every neural method moved
    # < 1.5 ASR points).
    t = float(np.quantile(cal, 1 - budget))
    if (cal >= t).mean() > budget:
        above = cal[cal > t]
        t = float(above.min()) if len(above) else float(np.nextafter(cal.max(), np.inf))
    return t

def nested(inj, ben, extra=None, n_rep=N_THRESH, seed=0):
    inj = np.asarray(inj); inj = inj[~np.isnan(inj)]
    ben = np.asarray(ben); ben = ben[~np.isnan(ben)]
    if len(ben) < 2 or len(inj) == 0: return None
    rng = np.random.default_rng(seed); half = max(1, len(ben)//2)
    ts, fp, asr = [], [], []
    ex = {k: [] for k in (extra or {})}
    for _ in range(n_rep):
        i = rng.permutation(len(ben)); t = thr(ben[i[:half]])
        ts.append(t); fp.append((ben[i[half:]] >= t).mean()*100); asr.append((inj < t).mean()*100)
        for k, v in (extra or {}).items():
            v = np.asarray(v); v = v[~np.isnan(v)]
            ex[k].append((v >= t).mean()*100 if len(v) else np.nan)
    q = lambda a: [float(np.mean(a)), float(np.percentile(a,2.5)), float(np.percentile(a,97.5))]
    y = np.r_[np.zeros(len(ben)), np.ones(len(inj))]
    lo, hi = wilson(int(round(np.mean(asr)/100*len(inj))), len(inj))
    return {"auc": float(roc_auc_score(y, np.r_[ben, inj])),
            "ap": float(average_precision_score(y, np.r_[ben, inj])),
            "asr": q(asr), "asr_binom": [lo, hi], "fpr": q(fp), "threshold": q(ts),
            "cross": {k: q(v) for k, v in ex.items()}, "n_inj": len(inj), "n_ben": len(ben)}

PAIR = {"SPML": "SPML-Benign", "DeepSet": "DeepSet-Benign",
        "BIPIA": "BIPIA-Clean", "InjecAgent": "InjecAgent-Clean"}
HARD = {"BIPIA": "BIPIA-HardNeg", "InjecAgent": "InjecAgent-HardNeg"}
cor = S.corpus.values

def evaluate(col):
    s = S[col].values
    out, ti, tm = {}, 0, 0.0
    for ic, bc in PAIR.items():
        extra = {}
        if ic in HARD: extra["hardneg"] = s[cor == HARD[ic]]
        pt = nested(s[cor == ic], s[cor == bc], extra=extra)
        if pt is None: continue
        out[ic] = pt; ti += pt["n_inj"]; tm += pt["asr"][0]/100*pt["n_inj"]
    out["_pooled_asr"] = float(tm/ti*100) if ti else float("nan")
    out["_n"] = int(ti)
    return out

def mcnemar(a_col, b_col, fam):
    """Paired test on the SAME injections, each method at its OWN 10%-FPR threshold (as v17b Exp 1b)."""
    ea, eb = E[a_col], E[b_col]
    if fam not in ea or fam not in eb: return None
    idx = np.where(cor == fam)[0]
    a, b_ = S[a_col].values[idx], S[b_col].values[idx]
    ok = ~(np.isnan(a) | np.isnan(b_)); a, b_ = a[ok], b_[ok]
    da = a >= ea[fam]["threshold"][0]; db = b_ >= eb[fam]["threshold"][0]
    oa, ob = int((da & ~db).sum()), int((~da & db).sum())
    p = binomtest(min(oa, ob), oa + ob, 0.5).pvalue if oa + ob else 1.0
    return {"only_a": oa, "only_b": ob, "p": float(p),
            "winner": ("a" if oa > ob else "b") if p < 0.05 else "neither"}
print("protocol functions loaded (identical to v17b)")

In [ ]:
# ---- runner: every trained method goes through THIS, on the SAME splits ----------------------
TIMES = {}
def run(names, fit_predict, protocols=("P1", "P2", "P2p", "LOCO")):
    """fit_predict(tr, te) -> {name: scores for te}. Everything that learns (PCA, vocabulary, scaler,
    head, prototypes) is fitted INSIDE it, on tr only. Scores are cached per name__protocol.npy, so a
    re-run skips what exists."""
    for P in protocols:
        paths = {n: f"{SC}/{n}__{P}.npy" for n in names}
        if all(os.path.exists(p) for p in paths.values()):
            for n, p in paths.items(): S[f"{n}@{P}"] = np.load(p)
            print(f"  {P:<5} loaded from disk: {names}"); continue
        out = {n: np.full(len(ev), np.nan) for n in names}
        t0 = time.perf_counter()
        for k, (tr, te) in enumerate(PROTOCOLS[P]):
            ts = time.perf_counter()
            res = fit_predict(tr, te)
            for n in names: out[n][te] = res[n]
            print(f"    {P:<5} split {k+1}/{len(PROTOCOLS[P])}  train {len(tr):>6,}  test {len(te):>6,}"
                  f"  {time.perf_counter()-ts:6.1f}s", flush=True)
        dt = time.perf_counter() - t0
        for n in names:
            a = out[n]
            if P == "LOCO":   # the 4 families complete; WJB/XSTest/JBB never held out -> NaN BY DESIGN
                assert not np.isnan(a[FAM_ROWS]).any() and np.isnan(a[~FAM_ROWS]).all(), f"{n}@{P}: LOCO gaps"
            else:
                assert not np.isnan(a).any(), f"{n}@{P}: gaps in scores"
            np.save(paths[n], a.astype(np.float32)); S[f"{n}@{P}"] = a.astype(np.float32)
            ok = ~np.isnan(a)
            print(f"  {P:<5} {n:<14} AUC(all scored rows) {roc_auc_score(y[ok], a[ok]):.4f} | {dt:.0f}s")
        TIMES[(",".join(names), P)] = dt
print("runner ready. Protocols:", {P: len(v) for P, v in PROTOCOLS.items()}, "splits")

In [ ]:
PCA_SEED = 42
# ================= MSLSAM-P: a probe directly on the hidden states ===========================
# No prototypes, no Mahalanobis, no 24-number summary. Per layer PCA(256) fitted on the TRAINING
# fold only, concatenated (8 x 256 = 2,048), standardised, then a logistic-regression probe whose C
# is chosen by an INNER grouped CV on the training fold. The test fold never influences anything.
RAW = {}
for li in LAYERS:
    RAW[li] = np.load(f"{CACHE}/layer_{li}.npy")[ev_rows].astype(np.float32)
H = RAW[LAYERS[0]].shape[1]
print(f"loaded {len(LAYERS)} layers x {RAW[LAYERS[0]].shape} "
      f"({sum(a.nbytes for a in RAW.values())/1e9:.2f} GB)")

CHOSEN_C, LAST = [], {}
def probe_features(tr, te):
    k = min(PROBE_DIMS, len(tr) - 1, H)
    pcas, Ftr, Fte = [], [], []
    for li in LAYERS:
        p = PCA(n_components=k, random_state=PCA_SEED).fit(RAW[li][tr])   # v18b: the ONLY varied seed
        pcas.append(p); Ftr.append(p.transform(RAW[li][tr])); Fte.append(p.transform(RAW[li][te]))
    Ftr, Fte = np.hstack(Ftr).astype(np.float32), np.hstack(Fte).astype(np.float32)
    sc = StandardScaler().fit(Ftr)
    return sc.transform(Ftr), sc.transform(Fte), pcas, sc

def pick_C(F, tr):
    # inner CV on the training fold only, grouped by payload (G2) so the inner split cannot
    # reward payload memorisation either. (PCA/scaler were fitted on the whole training fold --
    # unsupervised, and it only affects which C is picked, never the test fold.)
    yy, gg = y[tr], G2[tr]
    inner = StratifiedGroupKFold(3, shuffle=True, random_state=SEED)
    folds = list(inner.split(F, yy, gg))
    score = {}
    for C in PROBE_C:
        aucs = []
        for a, b in folds:
            m = LogisticRegression(C=C, class_weight="balanced", max_iter=3000).fit(F[a], yy[a])
            aucs.append(roc_auc_score(yy[b], m.decision_function(F[b])))
        score[C] = float(np.mean(aucs))
    return max(score, key=score.get), score

def probe_fp(tr, te):
    Ftr, Fte, pcas, sc = probe_features(tr, te)
    C, sc_c = pick_C(Ftr, tr); CHOSEN_C.append({"C": C, "inner_auc": sc_c})
    lr = LogisticRegression(C=C, class_weight="balanced", max_iter=3000).fit(Ftr, y[tr])
    out = {"mslsam_p_lr": lr.predict_proba(Fte)[:, 1]}
    if RUN_HGB_PROBE:
        hgb = HistGradientBoostingClassifier(max_iter=300, random_state=SEED).fit(Ftr, y[tr])
        out["mslsam_p_hgb"] = hgb.predict_proba(Fte)[:, 1]
    LAST.update(pcas=pcas, sc=sc, lr=lr)
    return out

In [ ]:
# ================= MSLSAM v17: v17b's code (via v18), with the PCA seed as a parameter =========
def build_v17_pca(PCA_SEED_V17):
    global PC, MU, CINV, N_PCA
    PC, MU, CINV = {}, {}, {}
    for li in LAYERS:
        X = np.load(f"{CACHE}/layer_{li}.npy")
        N_PCA = min(PCA_DIMS, int(is_base.sum()) - 1, X.shape[1])
        p = PCA(n_components=N_PCA, random_state=PCA_SEED_V17).fit(X[is_base])   # randomized SVD here; v18 found the seed moves MSLSAM v17 ~1.3 pts in-dist, ~7 pts LOCO
        Z = p.transform(X).astype(np.float32); PC[li] = Z[ev_rows]
        b = p.transform(X[is_base]); MU[li] = b.mean(0)
        CINV[li] = np.linalg.inv(np.cov(b.T) + np.eye(N_PCA) * 1e-4)
        del X, Z, b
    print(f"PCA {N_PCA}/layer")

def _norm(A): return A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-8)
def max_cos_exact(q, pr, li):
    Sm = _norm(PC[li][q]) @ _norm(PC[li][pr]).T
    pos = {int(r): j for j, r in enumerate(pr)}
    for i_, r in enumerate(q):
        j = pos.get(int(r))
        if j is not None: Sm[i_, j] = -np.inf
    return Sm.max(axis=1)
def mahal(Z, li):
    d = Z - MU[li]; return np.sqrt(np.einsum("ij,jk,ik->i", d, CINV[li], d))
FAM = ["direct", "indirect"]
def features(rws, proto):
    cols = []
    for f in FAM:
        pr = proto.get(f, np.array([], int))
        for li in LAYERS:
            cols.append(np.zeros(len(rws), np.float32) if len(pr) == 0 else max_cos_exact(rws, pr, li))
    for li in LAYERS: cols.append(mahal(PC[li][rws], li))
    return np.column_stack(cols).astype(np.float32)

GK = "group_carrier"
def fit_score(tr_rows, te_rows):
    y = ev.label.values[tr_rows]
    proto = {f: tr_rows[(y == 1) & (ev.family.values[tr_rows] == f)] for f in FAM}
    Xtr, Xte = features(tr_rows, proto), features(te_rows, proto)
    sc = RobustScaler().fit(Xtr)
    c = HistGradientBoostingClassifier(max_iter=300, random_state=SEED).fit(sc.transform(Xtr), y)
    return c.predict_proba(sc.transform(Xte))[:, 1]

In [ ]:
# ================= run: integrity (seed 42) first, then seeds 0-4 =============================
# TF-IDF is deterministic -> reuse v18's scores rather than refit.
for P in ["P2", "P2p", "LOCO"]:
    S[f"tfidf_lr@{P}"] = np.load(f"results/v18/scores/tfidf_lr__{P}.npy")

def probe_run(s, protocols):
    global PCA_SEED
    PCA_SEED = s
    run([f"mslsam_p_lr_s{s}"], lambda tr, te: {f"mslsam_p_lr_s{s}": probe_fp(tr, te)["mslsam_p_lr"]}, protocols)

def v17_run(s, protocols):
    todo = [P for P in protocols if not os.path.exists(f"{SC}/mslsam_v17_s{s}__{P}.npy")]
    if todo: build_v17_pca(s)          # the Dolly PCA + Mahalanobis stats depend on the seed
    run([f"mslsam_v17_s{s}"], lambda tr, te: {f"mslsam_v17_s{s}": fit_score(tr, te)}, protocols)

# ---- integrity: seed 42 through THIS code must reproduce v18 exactly ----
print("=== integrity: seed 42 must reproduce v18 ===")
probe_run(42, ["P2"])
checks = [("mslsam_p_lr_s42@P2", "results/v18/scores/mslsam_p_lr__P2.npy")]
if RUN_V17:
    v17_run(42, ["P2"]); checks.append(("mslsam_v17_s42@P2", "results/v18/scores/mslsam_v17__P2.npy"))
for col, path in checks:
    d = float(np.abs(S[col].values - np.load(path)).max())
    print(f"  {col:<22} vs v18: max |diff| = {d:.2e}")
    assert d < 1e-4, f"{col} does not reproduce v18 -- something other than the seed changed; STOP"
print("  integrity OK: the PCA seed is the only thing that varies below.\n")

for s in SEEDS:
    print(f"=== seed {s} ===")
    probe_run(s, ["P2", "P2p"] + (["LOCO"] if RUN_LOCO else []))
    if RUN_V17:
        v17_run(s, ["P2"] + (["LOCO"] if RUN_LOCO else []))

In [ ]:
# ================= RESULTS + PRE-REGISTERED STABILITY VERDICT =================================
E = {}
def ev_(col):
    if col not in E: E[col] = evaluate(col)
    return E[col]
pa = lambda col: ev_(col)["_pooled_asr"]
ALL_SEEDS = [42] + SEEDS                     # 42 = v18's seed, reported but NOT counted
for P in ["P2", "P2p", "LOCO"]: ev_(f"tfidf_lr@{P}")

rows_out = []
for s in ALL_SEEDS:
    pc = lambda P: f"mslsam_p_lr_s{s}@{P}"
    if pc("P2") not in S or (s != 42 and pc("P2p") not in S): continue
    r = {"seed": s, "counted": s != 42}
    r["P_P2"], r["T_P2"] = pa(pc("P2")), pa("tfidf_lr@P2")
    r["margin_P2"] = r["T_P2"] - r["P_P2"]
    mc = {f: mcnemar(pc("P2"), "tfidf_lr@P2", f) for f in FAMS}
    r["wins"] = [f for f, m in mc.items() if m and m["winner"] == "a"]
    r["losses"] = [f for f, m in mc.items() if m and m["winner"] == "b"]
    if s == 42:   # v18's own P2p number for seed 42
        r["P_P2p"] = pa("mslsam_p_lr_s42@P2p") if "mslsam_p_lr_s42@P2p" in S else \
                     json.load(open("results/v18/metrics.json"))["verdict"]["numbers"]["mslsam_p_lr@P2p"]
    else:
        r["P_P2p"] = pa(pc("P2p"))
    r["T_P2p"] = pa("tfidf_lr@P2p")
    r["H2_core"] = r["margin_P2"] >= 2 and len(r["wins"]) >= 2 and len(r["losses"]) == 0
    r["H2_rob"] = r["P_P2p"] <= r["T_P2p"]
    r["H2"] = r["H2_core"] and r["H2_rob"]
    e2 = ev_(pc("P2"))
    r["SPML"], r["DeepSet"] = e2["SPML"]["asr"][0], e2["DeepSet"]["asr"][0]
    r["BIPIA"], r["InjecAgent"] = e2["BIPIA"]["asr"][0], e2["InjecAgent"]["asr"][0]
    r["P_LOCO"] = pa(pc("LOCO")) if pc("LOCO") in S else np.nan
    v = f"mslsam_v17_s{s}"
    r["V_P2"] = pa(f"{v}@P2") if f"{v}@P2" in S else np.nan
    r["V_LOCO"] = pa(f"{v}@LOCO") if f"{v}@LOCO" in S else np.nan
    r["H1"] = bool(r["P_P2"] <= r["V_P2"] - 2) if not np.isnan(r["V_P2"]) else None
    r["H3"] = bool(r["P_LOCO"] <= r["V_LOCO"] + 2) if not (np.isnan(r["P_LOCO"]) or np.isnan(r["V_LOCO"])) else None
    rows_out.append(r)
R = pd.DataFrame(rows_out)
if 42 in set(R.seed) and np.isnan(R.loc[R.seed == 42, "P_LOCO"]).all():
    _v = json.load(open("results/v18/metrics.json"))["verdict"]["numbers"]
    R.loc[R.seed == 42, "P_LOCO"] = _v["mslsam_p_lr@LOCO"]; R.loc[R.seed == 42, "V_LOCO"] = _v["mslsam_v17@LOCO"]

pd.set_option("display.width", 200)
show = ["seed", "P_P2", "T_P2", "margin_P2", "wins", "losses", "P_P2p", "T_P2p", "H2",
        "SPML", "DeepSet", "BIPIA", "InjecAgent", "V_P2", "P_LOCO", "V_LOCO"]
print("  pooled ASR % (lower is better). P = MSLSAM-P (LR), T = TF-IDF, V = MSLSAM v17. Seed 42 = v18, not counted.\n")
print(R[show].round(2).to_string(index=False))

C = R[R.counted]
n_h2 = int(C.H2.sum()); n = len(C)
print(f"\n  spread over the {n} new seeds (mean ± SD, [min, max]):")
for col, name in [("P_P2", "MSLSAM-P  P2"), ("P_P2p", "MSLSAM-P  P2p"), ("P_LOCO", "MSLSAM-P  LOCO"),
                  ("V_P2", "MSLSAM v17 P2"), ("V_LOCO", "MSLSAM v17 LOCO"), ("margin_P2", "H2 margin (T - P)"),
                  ("SPML", "MSLSAM-P SPML"), ("DeepSet", "MSLSAM-P DeepSet")]:
    x = C[col].astype(float).dropna()
    if len(x): print(f"    {name:<20}{x.mean():7.2f} ± {x.std(ddof=1) if len(x) > 1 else 0:5.2f}   [{x.min():.2f}, {x.max():.2f}]")

if n_h2 >= 4:   reading, go = "STABLE -- v18's H2 claim stands", True
elif n_h2 >= 2: reading, go = "SEED-DEPENDENT (fragile) -- report H2 as such; v18's pass was partly luck", False
else:           reading, go = "RETRACTED -- seed 42 was a lucky draw", False
print(f"\n--- PRE-REGISTERED VERDICT ---")
print(f"  H2 holds in {n_h2} of {n} new seeds  ->  H2 is {reading}")
print(f"  v19 (task-drift delta, ~7 h GPU): {'RECOMMENDED' if go else 'NOT justified by this evidence'}")
h1 = C.H1.dropna(); h3 = C.H3.dropna()
if len(h1): print(f"  (descriptive) H1 holds in {int(h1.sum())}/{len(h1)} seeds"
                  + (" -> v18's 'H1 does not hold' is SEED-DEPENDENT" if h1.sum() >= 4 else ""))
if len(h3): print(f"  (descriptive) H3 holds in {int(h3.sum())}/{len(h3)} seeds")
print(f"  (descriptive) SPML < 5% in {int((C.SPML < 5).sum())}/{n} seeds; DeepSet < 5% in {int((C.DeepSet < 5).sum())}/{n} seeds")

def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): return o.item()
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, float) and np.isnan(o): return None
    return o
json.dump(_clean({"version": "v18b", "run_at": time.strftime("%Y-%m-%d %H:%M"), "seeds_counted": SEEDS,
                  "per_seed": R.to_dict(orient="records"), "E": E,
                  "verdict": {"H2_seeds_holding": n_h2, "of": n, "reading": reading, "v19_recommended": go},
                  "times_s": {f"{k[0]}|{k[1]}": v for k, v in TIMES.items()}}),
          open(f"{RES}/metrics.json", "w"), indent=1)
R.to_csv(f"{RES}/per_seed.csv", index=False)
print(f"\nwrote {RES}/metrics.json + per_seed.csv")

fig, ax = plt.subplots(figsize=(8, 4.5))
for j, (col, name) in enumerate([("P_P2", "MSLSAM-P P2"), ("T_P2", "TF-IDF P2"), ("V_P2", "MSLSAM v17 P2"),
                                  ("P_P2p", "MSLSAM-P P2p"), ("T_P2p", "TF-IDF P2p"),
                                  ("P_LOCO", "MSLSAM-P LOCO"), ("V_LOCO", "MSLSAM v17 LOCO")]):
    x = R[col].astype(float)
    ax.scatter(np.full(len(x), j) + np.linspace(-0.12, 0.12, len(x)), x,
               c=["red" if s == 42 else "C0" for s in R.seed], s=25)
ax.set_xticks(range(7)); ax.set_xticklabels(["P P2", "TF P2", "v17 P2", "P P2p", "TF P2p", "P LOCO", "v17 LOCO"])
ax.set_ylabel("pooled ASR % (lower is better)"); ax.set_title("v18b: spread over PCA seeds (red = v18's seed 42)")
plt.tight_layout(); plt.savefig(f"{RES}/{FIG}_seed_spread.png", dpi=150); plt.show()
print("\nNEXT: tell Claude 'v18b done'.")